# WRDS Indicator Fetcher

This notebook connects to WRDS and pulls a curated set of macro‑financial
indicators (e.g. term spread, VIX, high‑yield OAS) **without you writing any SQL**.

**How to use**

1. ⚙️ If you’re in a fresh environment, run the *Install* cell first.  
2. ✏️ Edit the *Parameters* cell – start/end dates, list of indicators, output file.  
3. ▶️ Run the remaining cells in order.  
4. 📁 A tidy DataFrame is saved locally for your analysis.

*Tip*: set environment variables `WRDS_USERNAME` and `WRDS_PASSWORD`
so you don’t type them each time.

In [ ]:
!pip install --quiet pandas wrds tqdm python-dateutil

In [1]:
# Imports
import pandas as pd
import datetime as dt
import os
from dateutil import parser as dtparser
from tqdm import tqdm
import wrds

In [5]:
# ↕️ Parameters – customise as needed
def parse_datestr(s):
    return dtparser.parse(s).date()

start = parse_datestr('2019-01-01')      # <-- change me
end   = parse_datestr('2025-05-11')      # <-- change me
indicators = ['TERM_SPREAD', 'HY_OAS', 'VIX', 'EPU']   # <-- put keys here
out_path = 'indicators.csv'          # feather, parquet, or csv

In [3]:
# 🗺️ Indicator metadata dictionary
INDICATORS = {
    # --- Term structure ---
    'DGS3': dict(schema='fred', table='fred_md', column='dgs3', freq='D'),
    'DGS10': dict(schema='fred', table='fred_md', column='dgs10', freq='D'),
    'TERM_SPREAD': dict(expr='DGS10 - DGS3', depends=['DGS10','DGS3']),

    # --- Credit ---
    'BAA': dict(schema='fred', table='fred_md', column='baa', freq='D'),
    'AAA': dict(schema='fred', table='fred_md', column='aaa', freq='D'),
    'IG_SPREAD': dict(expr='BAA - AAA', depends=['BAA','AAA']),
    'HY_OAS': dict(schema='ice', table='us_corpmaster', column='bamlh0a0hym2', freq='D'),

    # --- Volatility & risk aversion ---
    'VIX': dict(schema='cboe', table='daily_indices', column='vixcls', freq='D'),
    'MOVE': dict(schema='ice', table='move_index', column='move', freq='D'),

    # --- Sentiment / uncertainty ---
    'EPU': dict(schema='policy_uncertainty', table='daily', column='us_epu', freq='D'),
}

In [8]:
# ⚙️ Utility helpers
def wrds_connect():
    print('Connecting to WRDS …')
    return wrds.Connection()            # will prompt if env vars absent

def build_table_map(keys):
    mapping = {}
    for k in keys:
        meta = INDICATORS[k]
        if 'column' in meta:
            mapping.setdefault((meta['schema'], meta['table']), set()).add(meta['column'])
        elif 'depends' in meta:
            for d in meta['depends']:
                m = INDICATORS[d]
                mapping.setdefault((m['schema'], m['table']), set()).add(m['column'])
    return {k:list(v) for k,v in mapping.items()}

def fetch_table(conn, schema, table, cols, start, end):
    cols_sql = ', '.join(['date'] + cols)
    sql = f"SELECT date, {', '.join(cols)} FROM {schema}.{table} WHERE date BETWEEN '{start}' AND '{end}'"
    return conn.raw_sql(sql, date_cols=['date']).set_index('date')

def merge(frames):
    return pd.concat(frames, axis=1).sort_index()

def compute_synthetic(df, keys):
    for k in keys:
        meta = INDICATORS[k]
        if 'expr' in meta:
            df[k.lower()] = df.eval(meta['expr'].lower())
    return df

def keep_columns(df, keys):
    cols = []
    for k in keys:
        meta = INDICATORS[k]
        if 'column' in meta:
            cols.append(meta['column'].lower())
        else:
            cols.append(k.lower())
    return df[cols]

In [10]:
db = wrds.Connection()

OperationalError: (psycopg2.OperationalError) connection to server at "wrds-pgdata.wharton.upenn.edu" (165.123.60.118), port 9737 failed: server closed the connection unexpectedly
	This probably means the server terminated abnormally
	before or while processing the request.

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# 🚀 Fetch data
frames = []
for (schema, table), cols in tqdm(build_table_map(indicators).items()):
    frames.append(fetch_table(conn, schema, table, cols, start, end))

df = merge(frames)
df = compute_synthetic(df, indicators)
df = keep_columns(df, indicators)
df.tail()

Connecting to WRDS …


OperationalError: (psycopg2.OperationalError) could not translate host name "wrds-pgdata.wharton.upenn.edu" to address: No such host is known. 

(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
# 💾 Save to disk
ext = os.path.splitext(out_path)[1].lower()
if ext == '.feather':
    df.reset_index().to_feather(out_path)
elif ext in {'.parquet', '.pq'}:
    df.reset_index().to_parquet(out_path)
else:
    df.to_csv(out_path, index=True)
print(f'Saved → {out_path}')